# Proyecto QTranslate LIFIA

## Quirk → OpenQASM 3.0 → Ecosistema objetivo

Este cuaderno jupyter permite la ejecución del flujo de traducción de circuitos y etapas complementarias de verificación (visuales, zxcalculo, etc).

Flujo de ejecución:

1. Descarga e instalación de librerías externas necesarias
2. Traducción de circuitos [Quirk](https://algassert.com/quirk) a OpenQASM 3.0, utilizado como lenguaje de "pivote".

    Carga de los algoritmos de prueba iniciales (15) a partir del JSON algorithms (path: 'algorithms.json')
    Además incorpora los populares (38) representando extensiones de los iniciales (path: 'popular_algorithms.json').

3. Generación de circuitos visuales en Quirk (path: `/output/run_DD_MM_YYYY__HH/imgs/circuits_quirk`)
4. Generación de circuitos visuales en OpenQasm (path: `/output/run_DD_MM_YYYY__HH/imgs/circuits_qasm`)
5. Establecimiento unificado de imágenes para comparación visual (path: `/output/run_DD_MM_YYYY__HH/compare`)
6. Migración al ecosistema/lenguaje objetivo (Qiskit, Braket o Pennylane)

> Todas las salidas se agrupan en `output/run_DD_MM_YYYY__HH/`. La estampa conserva fecha y **hora** (sin minutos): dentro de la misma hora se reutiliza el mismo directorio y se sobrescriben los archivos generados.


### Descarga de las librerías necesarias para la funcionalidad


In [1]:
import sys
import subprocess
import importlib.util

print('Python:', sys.executable)

packages = {
    'qiskit': 'qiskit',
    'matplotlib': 'matplotlib',
    'qiskit_aer': 'qiskit-aer',
    'pylatexenc': 'pylatexenc',
    'qiskit_qasm3_import': 'qiskit-qasm3-import',
    'pandas': 'pandas',
    'pyzx': 'pyzx',
}

detalle_info = ''
paquetes_instalados = 0

for module_name, pip_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])
        estado = 'no existente, se instaló correctamente'
        paquetes_instalados += 1
    else:
        estado = 'existente previamente, no se instaló'

    info = f"{pip_name} - {estado}"
    detalle_info += (', ' if detalle_info else '') + info
    print(f'Paquete: {info}')

print(f'\nℹ️ Cantidad de paquetes instalados en este kernel: {paquetes_instalados}')
print('\n✅ Paquetes requeridos para la funcionalidad, disponibles actualmente en el kernel.')


Python: c:\Users\PC\desarrollo\QTransLIFIA\.venv\Scripts\python.exe
Paquete: qiskit - existente previamente, no se instaló
Paquete: matplotlib - existente previamente, no se instaló
Paquete: qiskit-aer - existente previamente, no se instaló
Paquete: pylatexenc - existente previamente, no se instaló
Paquete: qiskit-qasm3-import - existente previamente, no se instaló
Paquete: pandas - existente previamente, no se instaló
Paquete: pyzx - existente previamente, no se instaló

ℹ️ Cantidad de paquetes instalados en este kernel: 0

✅ Paquetes requeridos para la funcionalidad, disponibles actualmente en el kernel.


### Verificación de existencia y localización de archivos (.json) de circuitos

#### Generación de la estructura de salida (subdirectorios de /output)

Generamos una tabla (formato: MarkDown) por cada algoritmo de prueba que vayamos a ejecutar (paths: `/input/algorithms.json`, `/input/popular_algorithms.json` y `/input/qft_quirk.json`)


In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown
import os, sys
from datetime import datetime

# La raíz del proyecto se resuelve desde la ubicación del cuadernillo y no desde el
# directorio de trabajo actual: así ``input/`` y ``output/`` apuntan siempre a la
# raíz del proyecto aunque QTrans.ipynb se ejecute desde otra carpeta.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd() / 'notebooks')
        if (candidate / 'utils' / 'qutils.py').is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontró utils/qutils.py; ejecutá el cuadernillo dentro del proyecto')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

from utils.qutils import quirk_to_qasm, quirk_circuit_info
from utils.url_utils import parse_quirk_url, encode_quirk_url

# Constantes de directorios de entrada y salida del proyecto (siempre en la raíz del proyecto)
INPUT_DIR = PROJECT_ROOT / 'input'
OUTPUT_DIR = PROJECT_ROOT / 'output'
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'

# Etiqueta de la corrida por fecha y hora (sin minutos): /output/run_DD_MM_YYYY__HH
# Dentro de la misma hora se reutiliza el directorio y se sobrescriben los archivos.
RUN_LABEL = datetime.now().strftime('%d_%m_%Y__%H')

# Rutas de salida para la corrida actual (siempre en la raíz del proyecto). La
# estructura de carpetas NO se crea aquí: cada celda crea, bajo demanda, el
# directorio que necesita o genera (con Path.mkdir(parents=True, exist_ok=True)).
RUN_DIR = OUTPUT_DIR / f'run_{RUN_LABEL}'
RUN_QASM_DIR = RUN_DIR / 'algorithms_qasm'
RUN_IMGS_DIR = RUN_DIR / 'imgs'
RUN_CIRCUITS_DIR = RUN_DIR / 'migrated_circuits'
RUN_ZX_DIR = RUN_DIR / 'ZXCalculus'
RUN_COMPARE_DIR = RUN_IMGS_DIR / 'visual_comparations'

print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR}')
print(f"ESTAMPA (fecha y hora): {RUN_LABEL}  -> durante esta hora se reutiliza el directorio y se sobrescriben los archivos.")


def find_input_dir():
    """Directorio de entrada del proyecto (``<raíz>/input``), sin depender del cwd."""
    if (INPUT_DIR / "algorithms.json").is_file() and (INPUT_DIR / "popular_algorithms.json").is_file():
        return INPUT_DIR
    raise FileNotFoundError(f"No se encontró {INPUT_DIR / 'algorithms.json'} ni {INPUT_DIR / 'popular_algorithms.json'}")


def load_algorithms():
    path = find_input_dir() / "algorithms.json"
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_popular_algorithms():
    path = find_input_dir() / "popular_algorithms.json"
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_qft_algorithms():
    path = find_input_dir() / "qft_quirk.json"
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

# Algoritmos presentes en /input
MAIN_ALGORITHMS = load_algorithms()
POPULAR_ALGORITHMS = load_popular_algorithms()
QFT_ALGORITHMS = load_qft_algorithms()


def describe_algorithms(algorithms, title):
    md = f"## {title}\n\n| # | Circuito | Descripción | # Qubits |\n|---|---|---|---:|\n"
    for index, (key, value) in enumerate(algorithms.items(), 1):
        info = quirk_circuit_info(value["url"])
        n_qubits = info["n_qubits"]
        algorithm_name = key.split(". ", 1)[1] if ". " in key else key
        number = key.split(". ", 1)[0] if ". " in key else str(index)
        md += f'| {number} | **{algorithm_name}** | {value["desc"]} | {n_qubits} |\n'
    display(Markdown(md))
    print(f"ℹ️ Se detectaron {len(algorithms)} circuitos de entrada.")


describe_algorithms(MAIN_ALGORITHMS, "Algoritmos principales")
describe_algorithms(POPULAR_ALGORITHMS, "Popular algorithms")
describe_algorithms(QFT_ALGORITHMS, "Algoritmos QFT")

total_algoritmos = len(MAIN_ALGORITHMS) + len(POPULAR_ALGORITHMS) + len(QFT_ALGORITHMS)
print(f"ℹ️ Total de circuitos de entrada: {total_algoritmos} (principales + populares + QFT).")


DIRECTORIO DE LA CORRIDA: c:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20
ESTAMPA (fecha y hora): 14_10_2026__20  -> durante esta hora se reutiliza el directorio y se sobrescriben los archivos.


## Algoritmos principales

| # | Circuito | Descripción | # Qubits |
|---|---|---|---:|
| 1 | **Shor** | Algoritmo de factorización de Shor (4 qubits) | 4 |
| 2 | **Bernstein-Vazirani** | Algoritmo de Bernstein-Vazirani (4 qubits) | 4 |
| 3 | **Grover** | Algoritmo de búsqueda de Grover (2 qubits) | 2 |
| 4 | **Deutsch-Jozsa** | Algoritmo de Deutsch-Jozsa (4 qubits) | 4 |
| 5 | **Simon** | Algoritmo de Simon (6 qubits) | 6 |
| 6 | **TSP** | Circuito para problema del viajante (TSP, 3 qubits) | 3 |
| 7 | **Teleportation** | Protocolo de teleportación cuántica (5 qubits) | 5 |
| 8 | **Phase Estimation** | Estimación de fase cuántica (4 qubits + compuertas custom) | 4 |
| 9 | **QFT** | Transformada de Fourier Cuántica (3 qubits) | 3 |
| 10 | **QAOA** | Quantum Approximate Optimization Algorithm (2 qubits) | 2 |
| 11 | **Kickback** | Phase kickback (2 qubits) | 2 |
| 12 | **Full Adder** | Sumador completo (4 qubits) | 4 |
| 13 | **Multicontroled Gates** | Circuito de prueba con compuertas multicontroladas CCX, CCY, CCZ, CCCX, CCCY y CCCZ | 5 |
| 14 | **Shor_6q** | Algoritmo de factorización de Shor (6 qubits) | 6 |
| 15 | **Popurri_6q** | Algoritmo combinado para pruebas multicontroladas (6 qubits) | 6 |


ℹ️ Se detectaron 15 circuitos de entrada.


## Popular algorithms

| # | Circuito | Descripción | # Qubits |
|---|---|---|---:|
| 1 | **adder_n4_vq** | Circuito adder_n4_vq (4 qubits) | 4 |
| 2 | **adder_n10_vq** | Circuito adder_n10_vq (10 qubits) | 10 |
| 3 | **adder_n13_vq** | Circuito adder_n13_vq (13 qubits) | 13 |
| 4 | **adder_n16_vq** | Circuito adder_n16_vq (16 qubits) | 16 |
| 5 | **bv_2_vq** | Circuito bv_2_vq (2 qubits) | 2 |
| 6 | **bv_3_vq** | Circuito bv_3_vq (3 qubits) | 3 |
| 7 | **bv_4_vq** | Circuito bv_4_vq (4 qubits) | 4 |
| 8 | **bv_5_vq** | Circuito bv_5_vq (5 qubits) | 5 |
| 9 | **bv_9_vq** | Circuito bv_9_vq (9 qubits) | 9 |
| 10 | **bv_14_vq** | Circuito bv_14_vq (14 qubits) | 14 |
| 11 | **dj_indep_4_mqt** | Circuito dj_indep_4_mqt (4 qubits) | 4 |
| 12 | **dj_indep_5_mqt** | Circuito dj_indep_5_mqt (5 qubits) | 5 |
| 13 | **dj_indep_7_mqt** | Circuito dj_indep_7_mqt (7 qubits) | 7 |
| 14 | **dj_indep_14_mqt** | Circuito dj_indep_14_mqt (14 qubits) | 14 |
| 15 | **grover-noancilla_3_mqt** | Circuito grover-noancilla_3_mqt (3 qubits) | 3 |
| 16 | **grover-noancilla_4_mqt** | Circuito grover-noancilla_4_mqt (4 qubits) | 4 |
| 17 | **grover-v-chain_3_mqt** | Circuito grover-v-chain_3_mqt (3 qubits) | 3 |
| 18 | **grover-v-chain_4_mqt** | Circuito grover-v-chain_4_mqt (4 qubits) | 4 |
| 19 | **grover_3_vq** | Circuito grover_3_vq (3 qubits) | 3 |
| 20 | **grover_7_vq** | Circuito grover_7_vq (7 qubits) | 7 |
| 21 | **grover_9_vq** | Circuito grover_9_vq (9 qubits) | 9 |
| 22 | **kickback_7_vq** | Circuito kickback_7_vq (7 qubits) | 7 |
| 23 | **pe_2_vq** | Circuito pe_2_vq (3 qubits) | 3 |
| 24 | **pe_3_mqt** | Circuito pe_3_mqt (3 qubits) | 3 |
| 25 | **pe_4_mqt** | Circuito pe_4_mqt (4 qubits) | 4 |
| 26 | **pe_5_mqt** | Circuito pe_5_mqt (5 qubits) | 5 |
| 27 | **pe_6_mqt** | Circuito pe_6_mqt (6 qubits) | 6 |
| 28 | **pe_7_mqt** | Circuito pe_7_mqt (7 qubits) | 7 |
| 29 | **qft_2_mqt** | Circuito qft_2_mqt (2 qubits) | 2 |
| 30 | **qft_3_mqt** | Circuito qft_3_mqt (3 qubits) | 3 |
| 31 | **qft_4_mqt** | Circuito qft_4_mqt (4 qubits) | 4 |
| 32 | **qft_6_mqt** | Circuito qft_6_mqt (6 qubits) | 6 |
| 33 | **qft_7_vq** | Circuito qft_7_vq (7 qubits) | 7 |
| 34 | **shor_mod15_mqt** | Circuito shor_mod15_mqt (5 qubits) | 5 |
| 35 | **shor_mod21_mqt** | Circuito shor_mod21_mqt (8 qubits) | 8 |
| 36 | **simon_vq** | Circuito simon_vq (6 qubits) | 6 |
| 37 | **tsp_indep_4_mqt** | Circuito tsp_indep_4_mqt (4 qubits) | 4 |
| 38 | **tsp_indep_5_mqt** | Circuito tsp_indep_5_mqt (5 qubits) | 5 |


ℹ️ Se detectaron 38 circuitos de entrada.


### Captura de screenshots de circuitos originales Quirk

Generación de archivos de circuitos en el path: `.../imgs/circuits_quirk`


In [ ]:
import base64
import subprocess
import sys

try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
except ImportError:
    print("Instalando selenium...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'selenium'])
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

from pathlib import Path
import os
import time
import re

visual_root = RUN_IMGS_DIR

quirk_original_dir = visual_root / 'circuits_quirk'
quirk_original_dir.mkdir(parents=True, exist_ok=True)
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

def natural_sort_key(value):
    text = str(value).strip()
    match = re.search(r'^(\d+)', text)
    if match:
        num = int(match.group(1))
        return (num, text)
    else:
        return (float('inf'), text)

def capture_quirk_circuit(url, output_path):
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')

    driver = None
    try:
        driver = webdriver.Chrome(options=options)
        driver.get(url)

        wait = WebDriverWait(driver, 10)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, 'canvas')))
        time.sleep(2)

        page_size = driver.execute_script("""
            const root = document.documentElement;
            const body = document.body;
            return {
                width: Math.max(root.scrollWidth, body.scrollWidth, root.clientWidth),
                height: Math.max(root.scrollHeight, body.scrollHeight, root.clientHeight)
            };
        """)
        screenshot = driver.execute_cdp_cmd('Page.captureScreenshot', {
            'format': 'png',
            'fromSurface': True,
            'captureBeyondViewport': True,
            'clip': {
                'x': 0,
                'y': 0,
                'width': page_size['width'],
                'height': page_size['height'],
                'scale': 1,
            },
        })
        Path(output_path).write_bytes(base64.b64decode(screenshot['data']))
        return True
    except Exception as exc:
        print(f"  Error capturando {url}: {exc}")
        return False
    finally:
        if driver:
            driver.quit()

print('Captura de screenshots de circuitos originales desde Quirk\n')

ALGORITHMS = load_algorithms()
POPULAR_ALGORITHMS = load_popular_algorithms()
QFT_ALGORITHMS = load_qft_algorithms()

captured = 0
total_circuits = len(ALGORITHMS) + len(POPULAR_ALGORITHMS) + len(QFT_ALGORITHMS)
algorithm_collections = [
    (ALGORITHMS, 'main_'),
    (POPULAR_ALGORITHMS, 'popular_'),
    (QFT_ALGORITHMS, 'qft_'),
]

# Recorre la colección de algoritmos
for algorithms, filename_prefix in algorithm_collections:
    for name, data in sorted(algorithms.items(), key=lambda item: natural_sort_key(item[0])):
        url = data['url']
        safe_name = name.replace('.', '_').replace(' ', '_')
        output_path = quirk_original_dir / f'{filename_prefix}{safe_name}.png'

        print(f'Capturando: {name}...', end=' ')
        if capture_quirk_circuit(url, output_path):
            print(f'✅ Guardado en {output_path.name}')
            captured += 1
        else:
            print('❌ Error')

print(f'\n✅ Etapa de capturas completada: {captured}/{total_circuits} capturas de circuitos generadas')
print(f'\nℹ️ Archivos disponibles en: {quirk_original_dir}')


DIRECTORIO DE LA CORRIDA: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20
Captura de screenshots de circuitos originales desde Quirk

Capturando: 1. Shor... ✅ Guardado en 1__Shor.png
Capturando: 2. Bernstein-Vazirani... ✅ Guardado en 2__Bernstein-Vazirani.png
Capturando: 3. Grover... ✅ Guardado en 3__Grover.png
Capturando: 4. Deutsch-Jozsa... ✅ Guardado en 4__Deutsch-Jozsa.png
Capturando: 5. Simon... ✅ Guardado en 5__Simon.png
Capturando: 6. TSP... ✅ Guardado en 6__TSP.png
Capturando: 7. Teleportation... ✅ Guardado en 7__Teleportation.png
Capturando: 8. Phase Estimation... ✅ Guardado en 8__Phase_Estimation.png
Capturando: 9. QFT... ✅ Guardado en 9__QFT.png
Capturando: 10. QAOA... ✅ Guardado en 10__QAOA.png
Capturando: 11. Kickback... ✅ Guardado en 11__Kickback.png
Capturando: 12. Full Adder... ✅ Guardado en 12__Full_Adder.png
Capturando: 13. Multicontroled Gates... ✅ Guardado en 13__Multicontroled_Gates.png
Capturando: 14. Shor_6q... ✅ Guardado en 14__Shor_6q.png
Capturand

# Traducción de circuitos a OpenQASM


### Generación de circuitos OpenQASM

Traducción de circuitos Quirk --> OpenQASM
En formato textual (archivos .txt)

La versión de OpenQASM puede ser: 3.0 o 2.0 dependiendo de la parametrización.

Archivos generados en el path: `.../algorithms_qasm`


In [4]:
import os
import re
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# Versión de OpenQASM a generar: configurable en .env (QTRANS_QASM_VERSION);
# por defecto '3.0', acepta '2.0'.
load_dotenv()
QASM_VERSION = (os.getenv("QTRANS_QASM_VERSION") or "3.0").strip()
if QASM_VERSION not in {'2.0', '3.0'}:
    raise ValueError(f"QASM_VERSION debe ser '2.0' o '3.0' (recibido: {QASM_VERSION!r})")
QASM_VERSION_SUFFIX = f'_v{QASM_VERSION}'


def _env_bool(name, default=True):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().casefold() in {'1', 'true', 'yes', 'on'}


# Verbosidad y activación del workflow ZX (mismos parámetros que el pipeline CLI).
VERBOSE = _env_bool('QTRANS_VERBOSE', True)
GENERATE_ZX = _env_bool('QTRANS_GENERATE_ZX', True)


def strip_qasm_version(stem):
    """Quita el sufijo _vX.Y del nombre; las celdas posteriores lo usan para
    derivar los artefactos (imágenes, grafos, scripts) sin el sufijo de versión."""
    return re.sub(r'_v\d+\.\d+$', '', stem)


# Directorio versionado por corrida: /output/run_DD_MM_YYYY__HH/algorithms_qasm
output_dir = RUN_QASM_DIR
output_dir.mkdir(parents=True, exist_ok=True)
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')
print(f'VERSIÓN DE OpenQASM: {QASM_VERSION}')


def generate_qasm_table_and_files():
    rows = []
    generated_paths = []
    algorithm_collections = [
        ("Algoritmos Principales", ALGORITHMS, "main_"),
        ("Algoritmos populares", POPULAR_ALGORITHMS, "popular_"),
        ("Algoritmos QFT", QFT_ALGORITHMS, "qft_"),
    ]

    for collection_name, algorithms, filename_prefix in algorithm_collections:
        for name, data in algorithms.items():
            url = data["url"]
            info = quirk_circuit_info(url)
            qasm_code = quirk_to_qasm(url, data.get("offset", 0), qasm_version=QASM_VERSION)

            safe_name = name.replace(".", "_").replace(" ", "_")
            file_path = output_dir / f"{filename_prefix}{safe_name}{QASM_VERSION_SUFFIX}.txt"
            file_path.write_text(qasm_code, encoding="utf-8")
            generated_paths.append(file_path.resolve())

            rows.append({
                "Colección": collection_name,
                "Algoritmo": name.split(". ", 1)[1] if ". " in name else name,
                "Qubits": info["n_qubits"],
                "Columnas": info["n_cols"],
                "Descripcion": data.get("desc", ""),
                "Archivo": str(file_path.name),
                "Líneas": len(qasm_code.splitlines())
            })

    total_archivos = len(generated_paths)
    print(f"\n ✅ Todos los archivos generados en el path: {output_dir.resolve()} ({total_archivos} archivos)")
    for generated_path in generated_paths:
        print(f"- {generated_path.name}")

    df = pd.DataFrame(rows)
    return df


generate_qasm_table_and_files()


DIRECTORIO DE LA CORRIDA: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20
VERSIÓN DE OpenQASM: 3.0

 ✅ Todos los archivos generados en el path: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20\algorithms_qasm (53 archivos)
- 1__Shor_v3.0.txt
- 2__Bernstein-Vazirani_v3.0.txt
- 3__Grover_v3.0.txt
- 4__Deutsch-Jozsa_v3.0.txt
- 5__Simon_v3.0.txt
- 6__TSP_v3.0.txt
- 7__Teleportation_v3.0.txt
- 8__Phase_Estimation_v3.0.txt
- 9__QFT_v3.0.txt
- 10__QAOA_v3.0.txt
- 11__Kickback_v3.0.txt
- 12__Full_Adder_v3.0.txt
- 13__Multicontroled_Gates_v3.0.txt
- 14__Shor_6q_v3.0.txt
- 15__Popurri_6q_v3.0.txt
- popular_1__adder_n4_vq_v3.0.txt
- popular_2__adder_n10_vq_v3.0.txt
- popular_3__adder_n13_vq_v3.0.txt
- popular_4__adder_n16_vq_v3.0.txt
- popular_5__bv_2_vq_v3.0.txt
- popular_6__bv_3_vq_v3.0.txt
- popular_7__bv_4_vq_v3.0.txt
- popular_8__bv_5_vq_v3.0.txt
- popular_9__bv_9_vq_v3.0.txt
- popular_10__bv_14_vq_v3.0.txt
- popular_11__dj_indep_4_mqt_v3.0.txt
- popular_12__dj_indep

,Colección,Algoritmo,Qubits,Columnas,Descripcion,Archivo,Líneas
0,Algoritmos Principales,Shor,4,13,Algoritmo de factorización de Shor (4 qubits),1__Shor_v3.0.txt,27
1,Algoritmos Principales,Bernstein-Vazirani,4,8,Algoritmo de Bernstein-Vazirani (4 qubits),2__Bernstein-Vazirani_v3.0.txt,19
2,Algoritmos Principales,Grover,2,8,Algoritmo de búsqueda de Grover (2 qubits),3__Grover_v3.0.txt,14
3,Algoritmos Principales,Deutsch-Jozsa,4,9,Algoritmo de Deutsch-Jozsa (4 qubits),4__Deutsch-Jozsa_v3.0.txt,23
4,Algoritmos Principales,Simon,6,8,Algoritmo de Simon (6 qubits),5__Simon_v3.0.txt,19
5,Algoritmos Principales,TSP,3,6,"Circuito para problema del viajante (TSP, 3 qu...",6__TSP_v3.0.txt,14
6,Algoritmos Principales,Teleportation,5,8,Protocolo de teleportación cuántica (5 qubits),7__Teleportation_v3.0.txt,14
7,Algoritmos Principales,Phase Estimation,4,17,Estimación de fase cuántica (4 qubits + compue...,8__Phase_Estimation_v3.0.txt,32
8,Algoritmos Principales,QFT,3,8,Transformada de Fourier Cuántica (3 qubits),9__QFT_v3.0.txt,16
9,Algoritmos Principales,QAOA,2,12,Quantum Approximate Optimization Algorithm (2 ...,10__QAOA_v3.0.txt,9


# Verificación visual de traducción de circuitos


### Generación de imágenes de circuitos traducidos OpenQASM3

Genera archivos en el path: `.../imgs/circuits_qasm`


In [5]:
from pathlib import Path
import os
import re
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.qasm3 import loads as qasm3_loads

if 'strip_qasm_version' not in globals():
    def strip_qasm_version(stem):
        return re.sub(r'_v\d+\.\d+$', '', stem)

visual_root = RUN_IMGS_DIR


circuits_dir = visual_root / 'circuits_qasm'
circuits_dir.mkdir(parents=True, exist_ok=True)
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')


def natural_sort_key(value):
    parts = re.split(r'[_\s.-]+', str(value))
    return [(0, int(p)) if p.isdigit() else (1, p.lower()) for p in parts if p]


def load_qasm_circuit(qasm_text: str):
    text = (qasm_text or '').strip()
    if not text:
        raise ValueError('El contenido QASM está vacío')

    upper = text.upper()
    if upper.startswith('OPENQASM 3.0'):
        return qasm3_loads(text)
    if upper.startswith('OPENQASM 2.0'):
        return QuantumCircuit.from_qasm_str(text)

    try:
        return qasm3_loads(text)
    except Exception:
        return QuantumCircuit.from_qasm_str(text)


def render_qasm_circuits():
    qasm_dir = RUN_QASM_DIR
    circuits_dir = visual_root / 'circuits_qasm'
    circuits_dir.mkdir(parents=True, exist_ok=True)

    if not qasm_dir.exists():
        raise FileNotFoundError(f'No existe la carpeta de QASM: {qasm_dir}')

    qasm_version_suffix = globals().get('QASM_VERSION_SUFFIX', '')
    files = sorted((p for p in qasm_dir.glob('*.txt') if not qasm_version_suffix or p.name.endswith(qasm_version_suffix + '.txt')), key=lambda p: natural_sort_key(p.stem))
    if not files:
        raise FileNotFoundError(f'No hay archivos .txt en {qasm_dir}')

    rendered = 0
    failed = 0
    for txt_path in files:
        qasm_text = txt_path.read_text(encoding='utf-8')

        try:
            circuit = load_qasm_circuit(qasm_text)
        except Exception as exc:
            print(f'No se pudo cargar {txt_path.name} ({type(exc).__name__}): {exc!r}')
            failed += 1
            continue

        png_path = circuits_dir / f'{strip_qasm_version(txt_path.stem)}.png'
        figure = None
        try:
            figure = circuit.draw(output='mpl', filename=str(png_path), style='bw')
            print(f'Generado: {png_path.name}')
            rendered += 1
        except Exception as exc:
            print(f'No se pudo dibujar {txt_path.name} ({type(exc).__name__}): {exc!r}')
            failed += 1
        finally:
            if figure is not None:
                plt.close(figure)

    print(f'\nℹ️ Resumen: {rendered}/{len(files)} imágenes generadas; {failed} archivos con error.')


print(f'\nℹ️ Generación de imágenes de circuitos traducidos a OpenQASM 3.0 ( {circuits_dir.resolve()} )\n')

render_qasm_circuits()


DIRECTORIO DE LA CORRIDA: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20

ℹ️ Generación de imágenes de circuitos traducidos a OpenQASM 3.0 ( C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20\imgs\circuits_qasm )

Generado: 1__Shor.png
Generado: 2__Bernstein-Vazirani.png
Generado: 3__Grover.png
Generado: 4__Deutsch-Jozsa.png
Generado: 5__Simon.png
Generado: 6__TSP.png
Generado: 7__Teleportation.png
Generado: 8__Phase_Estimation.png
Generado: 9__QFT.png
Generado: 10__QAOA.png
Generado: 11__Kickback.png
Generado: 12__Full_Adder.png
Generado: 13__Multicontroled_Gates.png
Generado: 14__Shor_6q.png
Generado: 15__Popurri_6q.png
Generado: popular_1__adder_n4_vq.png
Generado: popular_2__adder_n10_vq.png
Generado: popular_3__adder_n13_vq.png
Generado: popular_4__adder_n16_vq.png
Generado: popular_5__bv_2_vq.png
Generado: popular_6__bv_3_vq.png
Generado: popular_7__bv_4_vq.png
Generado: popular_8__bv_5_vq.png
Generado: popular_9__bv_9_vq.png
Generado: popular_10__bv_14_vq

### Comparación visual

Generación del subdirectorio para agilizar la verificación de equivalencia visual y la exactitud de la traducción de circuitos.
Establece la generación de imágenes generadas (circuitos quirk y OpenQASM)

Directorio de salida: `.../compare`


In [6]:
from pathlib import Path
import os
import re

compare_dir = RUN_COMPARE_DIR
compare_dir.mkdir(parents=True, exist_ok=True)
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

visual_root = RUN_IMGS_DIR

qasm_dir = visual_root / 'circuits_qasm'
quirk_dir = visual_root / 'circuits_quirk'
qasm_rel = os.path.relpath(qasm_dir, compare_dir).replace(os.sep, '/')
quirk_rel = os.path.relpath(quirk_dir, compare_dir).replace(os.sep, '/')

if not qasm_dir.exists():
    raise FileNotFoundError(f'No existe la carpeta de circuitos OpenQASM: {qasm_dir}')
if not quirk_dir.exists():
    raise FileNotFoundError(f'No existe la carpeta de circuitos Quirk: {quirk_dir}')


def natural_sort_key(value):
    parts = re.split(r'[_\s.-]+', str(value))
    return [(0, int(part)) if part.isdigit() else (1, part.lower()) for part in parts if part]


qasm_pngs = {path.stem: path for path in qasm_dir.glob('*.png')}
quirk_pngs = {path.stem: path for path in quirk_dir.glob('*.png')}
matching_names = sorted(set(qasm_pngs) & set(quirk_pngs), key=natural_sort_key)

index_lines = ['# Comparación de circuitos', '']

for stem in matching_names:
    qasm_path = qasm_pngs[stem]
    quirk_path = quirk_pngs[stem]
    title = stem.replace('_', ' ')
    md_path = compare_dir / f'{stem}.md'
    md_path.write_text(
        '\n'.join([
            f'# Comparación: {title}',
            '',
            '<table>',
            '<tr><th align="center">Circuito Original (Quirk)</th><th align="center">Circuito traducido (OpenQASM 3.0)</th></tr>',
            '<tr>',
            '<td align="center"><a href="' + quirk_rel + '/' + quirk_path.name + '" title="Abrir imagen original"><img src="' + quirk_rel + '/' + quirk_path.name + '" alt="' + quirk_path.name + '" width="100%" /></a></td>',
            '<td align="center"><a href="' + qasm_rel + '/' + qasm_path.name + '" title="Abrir imagen original"><img src="' + qasm_rel + '/' + qasm_path.name + '" alt="' + qasm_path.name + '" width="100%" /></a></td>',
            '</tr>',
            '</table>',
            '',
        ]),
        encoding='utf-8'
    )

    index_lines.append(f'- [{title}]({stem}.md)')
    print(f'Generado info comparativa --> {md_path.name}')

index_path = compare_dir / 'README.md'
index_path.write_text('\n'.join(index_lines) + '\n', encoding='utf-8')

print(f'\nℹ️ Carpeta de comparación creada en: {compare_dir.resolve()}')
print(f'\nℹ️ Se generaron {len(matching_names)} comparativas para archivos presentes en ambas carpetas.')
print(f'\nℹ️ Índice disponible en: {index_path.resolve()}')


DIRECTORIO DE LA CORRIDA: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20
Generado info comparativa --> 1__Shor.md
Generado info comparativa --> 2__Bernstein-Vazirani.md
Generado info comparativa --> 3__Grover.md
Generado info comparativa --> 4__Deutsch-Jozsa.md
Generado info comparativa --> 5__Simon.md
Generado info comparativa --> 6__TSP.md
Generado info comparativa --> 7__Teleportation.md
Generado info comparativa --> 8__Phase_Estimation.md
Generado info comparativa --> 9__QFT.md
Generado info comparativa --> 10__QAOA.md
Generado info comparativa --> 11__Kickback.md
Generado info comparativa --> 12__Full_Adder.md
Generado info comparativa --> 13__Multicontroled_Gates.md
Generado info comparativa --> 14__Shor_6q.md
Generado info comparativa --> 15__Popurri_6q.md
Generado info comparativa --> popular_1__adder_n4_vq.md
Generado info comparativa --> popular_2__adder_n10_vq.md
Generado info comparativa --> popular_3__adder_n13_vq.md
Generado info comparativa --> popular_4__a

# Verificación de equivalencia entre circuitos mediante ZXCalculus


### Generación de grafos ZX a partir del código Quirk (importado a un .json)

Directorio asociado: `.../ZXCalculus/algorithms_base`


In [7]:
import json
import re
import sys
from pathlib import Path

project_root = globals().get('PROJECT_ROOT') or next(
    (candidate for candidate in (Path.cwd(), Path.cwd().parent) if (candidate / 'utils' / 'qutils.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('No se encontró utils/qutils.py desde el directorio del notebook')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
from utils import qutils, zx_utils
importlib.reload(qutils)
importlib.reload(zx_utils)
quirk_to_zx_graph = zx_utils.quirk_to_zx_graph

algorithms_dir = project_root / 'input'
if (not (algorithms_dir / 'algorithms.json').is_file()
        or not (algorithms_dir / 'popular_algorithms.json').is_file()
        or not (algorithms_dir / 'qft_quirk.json').is_file()):
    raise FileNotFoundError('No se encontraron algorithms.json, popular_algorithms.json y qft_quirk.json dentro de input')

algorithm_collections = [
    (json.loads((algorithms_dir / 'algorithms.json').read_text(encoding='utf-8')), 'main_'),
    (json.loads((algorithms_dir / 'popular_algorithms.json').read_text(encoding='utf-8')), 'popular_'),
    (json.loads((algorithms_dir / 'qft_quirk.json').read_text(encoding='utf-8')), 'qft_'),
]
# Reutiliza la configuración de la corrida; si la celda se ejecuta aislada, la reconstruye
if 'RUN_ZX_DIR' not in globals():
    from datetime import datetime
    def _find_output_root():
        _root = globals().get('PROJECT_ROOT') or Path.cwd()
        for _candidate in (_root / 'output', Path.cwd() / 'output', Path.cwd().parent / 'output'):
            if _candidate.is_dir():
                return _candidate
        return _root / 'output'
    _output_root = _find_output_root()
    _existing_runs = sorted(
        (d for d in _output_root.glob('run_*') if d.is_dir()),
        key=lambda d: d.stat().st_mtime,
        reverse=True,
    )
    RUN_DIR = _existing_runs[0] if _existing_runs else _output_root / f"run_{datetime.now().strftime('%d_%m_%Y__%H')}"
    RUN_ZX_DIR = RUN_DIR / 'ZXCalculus'
output_dir = RUN_ZX_DIR / 'algorithms_base'
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')
quirk_time = 0.5

graph_files = {}
for collection, filename_prefix in algorithm_collections:
    for name, data in collection.items():
        graph = quirk_to_zx_graph(data['url'], data.get('offset', 0), time=quirk_time)
        graph_json = graph.to_json()
        json.loads(graph_json)
        safe_name = re.sub(r'[^A-Za-z0-9_-]+', '_', name).strip('_')
        filename = f'{filename_prefix}{safe_name}.json'
        if filename in graph_files:
            raise ValueError(f'Nombre de archivo duplicado: {filename}')
        graph_files[filename] = graph_json

expected_graphs = sum(len(collection) for collection, _ in algorithm_collections)
if len(graph_files) != expected_graphs:
    raise RuntimeError(f'Se esperaban {expected_graphs} grafos, se generaron {len(graph_files)}')

output_dir.mkdir(parents=True, exist_ok=True)
for filename, graph_json in graph_files.items():
    (output_dir / filename).write_text(graph_json, encoding='utf-8')

print(f'Grafo ZX generado para {len(graph_files)} algoritmos.')
print(f'Snapshot temporal de Quirk: {quirk_time} (aplica a gates dependientes del tiempo como Rxft).')
print(f'Archivos guardados en: {output_dir.resolve()}')


DIRECTORIO DE LA CORRIDA: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20
Grafo ZX generado para 53 algoritmos.
Snapshot temporal de Quirk: 0.5 (aplica a gates dependientes del tiempo como Rxft).
Archivos guardados en: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20\ZXCalculus\algorithms_base


### Generación de grafos ZX a partir del código OpenQasm generado a partir de nuestra migración

Directorio asociado: .../ZXCalculus/algorithms_qasm

Esta celda procesa únicamente los archivos `.txt` de la versión activa (`QASM_VERSION`, por defecto 3.0), identificada por el sufijo `_v3.0.txt` / `_v2.0.txt`.


In [8]:
import importlib
import json
import sys
from pathlib import Path

project_root = globals().get('PROJECT_ROOT') or next(
    (candidate for candidate in (Path.cwd(), Path.cwd().parent) if (candidate / 'utils' / 'qutils.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('No se encontró utils/qutils.py desde el directorio del notebook')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
from utils import qutils, zx_utils
importlib.reload(qutils)
importlib.reload(zx_utils)
qasm_to_zx_graph = zx_utils.qasm_to_zx_graph

if 'strip_qasm_version' not in globals():
    import re
    def strip_qasm_version(stem):
        return re.sub(r'_v\d+\.\d+$', '', stem)

# Reutiliza la configuración de la corrida; si la celda se ejecuta aislada, la reconstruye
if 'RUN_QASM_DIR' not in globals() or 'RUN_ZX_DIR' not in globals():
    from datetime import datetime
    def _find_output_root():
        _root = globals().get('PROJECT_ROOT') or Path.cwd()
        for _candidate in (_root / 'output', Path.cwd() / 'output', Path.cwd().parent / 'output'):
            if _candidate.is_dir():
                return _candidate
        return _root / 'output'
    _output_root = _find_output_root()
    _existing_runs = sorted(
        (d for d in _output_root.glob('run_*') if (d / 'algorithms_qasm').is_dir()),
        key=lambda d: d.stat().st_mtime,
        reverse=True,
    )
    RUN_DIR = _existing_runs[0] if _existing_runs else _output_root / f"run_{datetime.now().strftime('%d_%m_%Y__%H')}"
    RUN_QASM_DIR = RUN_DIR / 'algorithms_qasm'
    RUN_ZX_DIR = RUN_DIR / 'ZXCalculus'
qasm_dir = RUN_QASM_DIR
output_dir = RUN_ZX_DIR / 'algorithms_qasm'
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')
qasm_version_suffix = globals().get('QASM_VERSION_SUFFIX', '')
qasm_files = sorted(p for p in qasm_dir.glob('*.txt') if not qasm_version_suffix or p.name.endswith(qasm_version_suffix + '.txt'))
print(f'VERSIÓN DE OpenQASM procesada: {qasm_version_suffix or "(todos)"}')
if not qasm_files:
    raise FileNotFoundError(f'No hay archivos .txt para la versión activa {qasm_version_suffix!r} en {qasm_dir}')

graph_files = {}
for qasm_path in qasm_files:
    qasm_code = qasm_path.read_text(encoding='utf-8')
    graph = qasm_to_zx_graph(qasm_code)
    graph_json = graph.to_json()
    json.loads(graph_json)
    graph_files[f'{strip_qasm_version(qasm_path.stem)}.json'] = graph_json

expected_graphs = len({strip_qasm_version(p.stem) for p in qasm_files})
if len(graph_files) != expected_graphs:
    raise RuntimeError(f'Se esperaban {expected_graphs} grafos, se generaron {len(graph_files)}')

output_dir.mkdir(parents=True, exist_ok=True)
for filename, graph_json in graph_files.items():
    (output_dir / filename).write_text(graph_json, encoding='utf-8')

print(f'Grafos ZX generados: {len(graph_files)}.')
print(f'Circuitos OpenQASM leídos desde: {qasm_dir.resolve()}')
print(f'Grafos guardados en: {output_dir.resolve()}')


DIRECTORIO DE LA CORRIDA: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20
VERSIÓN DE OpenQASM procesada: _v3.0
Grafos ZX generados: 53.
Circuitos OpenQASM leídos desde: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20\algorithms_qasm
Grafos guardados en: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20\ZXCalculus\algorithms_qasm


### Verificación de Equivalencia - Comparación de grafos ZX (Quirk vs. OpenQASM)

Establecemos por cada algoritmo si la comparación de equivalencia es correcta, a fin de verificar la migración hacia OpenQASM.


In [9]:
import re
from pathlib import Path

import pandas as pd
import pyzx as zx
from IPython.display import display

zx_dir = RUN_ZX_DIR
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

base_dir = zx_dir / 'algorithms_base'
qasm_dir = zx_dir / 'algorithms_qasm'


def normalized_graph_name(path):
    return re.sub(r'_+', '_', path.stem).casefold()


def algorithm_order(path):
    stem = path.stem.casefold()
    is_popular = stem.startswith('popular_')
    name = stem.removeprefix('popular_') if is_popular else stem
    match = re.match(r'(\d+)', name)
    number = int(match.group(1)) if match else float('inf')
    return (is_popular, number, name)


def index_graphs(directory):
    paths = sorted(directory.glob('*.json'))
    if not paths:
        raise FileNotFoundError(f'No hay grafos JSON en {directory}')
    indexed = {}
    for path in paths:
        key = normalized_graph_name(path)
        if key in indexed:
            raise ValueError(f'Nombres ambiguos en {directory}: {indexed[key].name} y {path.name}')
        indexed[key] = path
    return indexed


base_graphs = index_graphs(base_dir)
qasm_graphs = index_graphs(qasm_dir)
missing_qasm = sorted(base_graphs.keys() - qasm_graphs.keys())
missing_base = sorted(qasm_graphs.keys() - base_graphs.keys())
if missing_qasm or missing_base:
    raise ValueError(f'No coinciden las colecciones. Sin QASM: {missing_qasm}; sin base: {missing_base}')

results = []
ordered_base_paths = sorted(base_graphs.values(), key=algorithm_order)
for base_path in ordered_base_paths:
    key = normalized_graph_name(base_path)
    qasm_path = qasm_graphs[key]
    base_graph = zx.Graph.from_json(base_path.read_text(encoding='utf-8'))
    qasm_graph = zx.Graph.from_json(qasm_path.read_text(encoding='utf-8'))
    base_qubits = base_graph.num_inputs()
    qasm_qubits = qasm_graph.num_inputs()

    base_circuit = None
    qasm_circuit = None
    if base_qubits == qasm_qubits and base_qubits > 10:
        base_circuit = zx.Circuit.from_graph(base_graph)
        qasm_circuit = zx.Circuit.from_graph(qasm_graph)

    zx.simplify.full_reduce(base_graph)
    zx.simplify.full_reduce(qasm_graph)

    if base_qubits != qasm_qubits:
        method = 'sin comparar'
        result = f'NO COMPARABLES (qubits distintos: {base_qubits} vs. {qasm_qubits})'
    elif base_qubits <= 10:
        method = 'compare_tensors (rw-greedy-linear)'
        equivalent = zx.compare_tensors(base_graph, qasm_graph, strategy='rw-greedy-linear')
        result = 'EQUIVALENTES' if equivalent else 'NO EQUIVALENTES'
    else:
        method = 'verify_equality'
        equivalent = base_circuit.verify_equality(qasm_circuit, up_to_global_phase=True)
        result = 'EQUIVALENTES' if equivalent else 'NO DEMOSTRADO'

    results.append({
        'Nombre algoritmo': base_path.stem,
        'Cant. cúbits': base_qubits,
        'Método utilizado': method,
        'Resultado': result,
    })
    if VERBOSE and GENERATE_ZX:
        print(f'[Comparación ZX] Quirk vs OpenQASM de {base_path.stem}: {result} (método: {method}).')

comparison_table = pd.DataFrame(
    results,
    columns=['Nombre algoritmo', 'Cant. cúbits', 'Método utilizado', 'Resultado'],
)
display(comparison_table)
if VERBOSE and GENERATE_ZX:
    equivalent = sum(1 for row in results if row['Resultado'].startswith('EQUIVALENTES'))
    not_demonstrated = sum(1 for row in results if row['Resultado'].startswith('NO DEMOSTRADO'))
    summary = f'[Comparación ZX] Grafos reducidos Quirk vs OpenQASM: {equivalent}/{len(results)} circuitos EQUIVALENTES'
    if not_demonstrated:
        summary += f', {not_demonstrated} NO DEMOSTRADOS'
    print(summary + '.')
print('Todos los grafos se redujeron en memoria antes de comparar; los JSON originales no se modificaron.')
print('Nota: NO DEMOSTRADO indica que verify_equality no pudo probar equivalencia; no demuestra inequivalencia.')


DIRECTORIO DE LA CORRIDA: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20
[Comparación ZX] Quirk vs OpenQASM de 1_Shor: EQUIVALENTES (método: compare_tensors (rw-greedy-linear)).
[Comparación ZX] Quirk vs OpenQASM de 2_Bernstein-Vazirani: EQUIVALENTES (método: compare_tensors (rw-greedy-linear)).
[Comparación ZX] Quirk vs OpenQASM de 3_Grover: EQUIVALENTES (método: compare_tensors (rw-greedy-linear)).
[Comparación ZX] Quirk vs OpenQASM de 4_Deutsch-Jozsa: EQUIVALENTES (método: compare_tensors (rw-greedy-linear)).
[Comparación ZX] Quirk vs OpenQASM de 5_Simon: EQUIVALENTES (método: compare_tensors (rw-greedy-linear)).
[Comparación ZX] Quirk vs OpenQASM de 6_TSP: EQUIVALENTES (método: compare_tensors (rw-greedy-linear)).
[Comparación ZX] Quirk vs OpenQASM de 7_Teleportation: EQUIVALENTES (método: compare_tensors (rw-greedy-linear)).
[Comparación ZX] Quirk vs OpenQASM de 8_Phase_Estimation: EQUIVALENTES (método: compare_tensors (rw-greedy-linear)).
[Comparación ZX] Quirk vs O

,Nombre algoritmo,Cant. cúbits,Método utilizado,Resultado
0,1_Shor,4,compare_tensors (rw-greedy-linear),EQUIVALENTES
1,2_Bernstein-Vazirani,4,compare_tensors (rw-greedy-linear),EQUIVALENTES
2,3_Grover,2,compare_tensors (rw-greedy-linear),EQUIVALENTES
3,4_Deutsch-Jozsa,4,compare_tensors (rw-greedy-linear),EQUIVALENTES
4,5_Simon,6,compare_tensors (rw-greedy-linear),EQUIVALENTES
5,6_TSP,3,compare_tensors (rw-greedy-linear),EQUIVALENTES
6,7_Teleportation,5,compare_tensors (rw-greedy-linear),EQUIVALENTES
7,8_Phase_Estimation,4,compare_tensors (rw-greedy-linear),EQUIVALENTES
8,9_QFT,3,compare_tensors (rw-greedy-linear),EQUIVALENTES
9,10_QAOA,2,compare_tensors (rw-greedy-linear),EQUIVALENTES


[Comparación ZX] Grafos reducidos Quirk vs OpenQASM: 53/53 circuitos EQUIVALENTES.
Todos los grafos se redujeron en memoria antes de comparar; los JSON originales no se modificaron.
Nota: NO DEMOSTRADO indica que verify_equality no pudo probar equivalencia; no demuestra inequivalencia.


### Generación de imágenes (grafos) asociados a los algoritmos base

Toma cada algoritmo presente en el directorio /algorithms_base y genera un .svg para comprobar la reducción.


In [10]:
import os
import re
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg', force=True)
import matplotlib.pyplot as plt
import pyzx as zx
from IPython.display import Markdown, display

zx.settings.drawing_backend = 'matplotlib'

zx_dir = RUN_ZX_DIR
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

notebook_dir = globals().get('NOTEBOOK_DIR') or (globals().get('PROJECT_ROOT') or Path.cwd()) / 'notebooks'

base_dir = zx_dir / 'algorithms_base'
output_dir = zx_dir / 'graphs'
graph_paths = list(base_dir.glob('*.json'))
if not graph_paths:
    raise FileNotFoundError(f'No hay grafos JSON en {base_dir}')


def graph_order(path):
    stem = path.stem.casefold()
    is_popular = stem.startswith('popular_')
    name = stem.removeprefix('popular_') if is_popular else stem
    match = re.match(r'(\d+)', name)
    number = int(match.group(1)) if match else float('inf')
    return is_popular, number, name


def markdown_escape(value):
    return str(value).replace('|', '\\|').replace('\n', ' ')


output_dir.mkdir(parents=True, exist_ok=True)
ordered_graph_paths = sorted(graph_paths, key=graph_order)
image_results = []
failed_count = 0

with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive')
    for index, graph_path in enumerate(ordered_graph_paths, start=1):
        graph = zx.Graph.from_json(graph_path.read_text(encoding='utf-8'))
        n_qubits = graph.num_inputs()
        vertices_before = graph.num_vertices()
        reduced_graph = graph.copy()
        figure = None
        image_path = output_dir / f'{graph_path.stem}.svg'

        try:
            zx.simplify.full_reduce(reduced_graph)
            vertices_after = reduced_graph.num_vertices()
            figure_width = max(12, min(24, vertices_after / 70))
            figure_height = max(5, min(12, n_qubits * 0.5))
            figure = zx.draw(reduced_graph, labels=True, figsize=(figure_width, figure_height))
            figure.savefig(image_path, format='svg')
            status = 'Generado'
        except Exception as exc:
            failed_count += 1
            vertices_after = reduced_graph.num_vertices()
            status = f'Error: {type(exc).__name__}'
        finally:
            if figure is not None:
                plt.close(figure)

        relative_path = Path(os.path.relpath(image_path, start=notebook_dir)).as_posix()
        image_reference = f'[Abrir SVG]({relative_path})' if image_path.is_file() else '-'
        image_results.append({
            'Nombre algoritmo': graph_path.stem,
            'Cúbits': n_qubits,
            'Vértices (antes → después)': f'{vertices_before} → {vertices_after}',
            'Referencia al grafo': image_reference,
            'Resultado': status,
        })

headers = ['Nombre algoritmo', 'Cúbits', 'Vértices (antes → después)', 'Referencia al grafo', 'Resultado']
markdown_rows = [
    '| ' + ' | '.join(headers) + ' |',
    '| ' + ' | '.join('---' for _ in headers) + ' |',
]
for row in image_results:
    cells = [
        markdown_escape(row['Nombre algoritmo']),
        str(row['Cúbits']),
        row['Vértices (antes → después)'],
        row['Referencia al grafo'],
        markdown_escape(row['Resultado']),
    ]
    markdown_rows.append('| ' + ' | '.join(cells) + ' |')

display(Markdown('\n'.join(markdown_rows)))
print(f'\nSVG generados: {len(image_results) - failed_count}/{len(ordered_graph_paths)}; errores: {failed_count}.')
print(f'Archivos guardados en: {output_dir.resolve()}')


DIRECTORIO DE LA CORRIDA: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20


| Nombre algoritmo | Cúbits | Vértices (antes → después) | Referencia al grafo | Resultado |
| --- | --- | --- | --- | --- |
| 1_Shor | 4 | 296 → 11 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/1_Shor.svg) | Generado |
| 2_Bernstein-Vazirani | 4 | 129 → 14 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/2_Bernstein-Vazirani.svg) | Generado |
| 3_Grover | 2 | 66 → 8 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/3_Grover.svg) | Generado |
| 4_Deutsch-Jozsa | 4 | 149 → 14 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/4_Deutsch-Jozsa.svg) | Generado |
| 5_Simon | 6 | 177 → 19 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/5_Simon.svg) | Generado |
| 6_TSP | 3 | 80 → 11 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/6_TSP.svg) | Generado |
| 7_Teleportation | 5 | 125 → 15 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/7_Teleportation.svg) | Generado |
| 8_Phase_Estimation | 4 | 239 → 28 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/8_Phase_Estimation.svg) | Generado |
| 9_QFT | 3 | 103 → 14 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/9_QFT.svg) | Generado |
| 10_QAOA | 2 | 58 → 4 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/10_QAOA.svg) | Generado |
| 11_Kickback | 2 | 56 → 7 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/11_Kickback.svg) | Generado |
| 12_Full_Adder | 4 | 326 → 54 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/12_Full_Adder.svg) | Generado |
| 13_Multicontroled_Gates | 5 | 1013 → 137 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/13_Multicontroled_Gates.svg) | Generado |
| 14_Shor_6q | 6 | 379 → 17 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/14_Shor_6q.svg) | Generado |
| 15_Popurri_6q | 6 | 4356 → 622 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/15_Popurri_6q.svg) | Generado |
| popular_1_adder_n4_vq | 4 | 284 → 54 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_1_adder_n4_vq.svg) | Generado |
| popular_2_adder_n10_vq | 10 | 848 → 156 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_2_adder_n10_vq.svg) | Generado |
| popular_3_adder_n13_vq | 13 | 1130 → 207 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_3_adder_n13_vq.svg) | Generado |
| popular_4_adder_n16_vq | 16 | 1412 → 258 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_4_adder_n16_vq.svg) | Generado |
| popular_5_bv_2_vq | 2 | 56 → 7 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_5_bv_2_vq.svg) | Generado |
| popular_6_bv_3_vq | 3 | 95 → 11 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_6_bv_3_vq.svg) | Generado |
| popular_7_bv_4_vq | 4 | 134 → 14 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_7_bv_4_vq.svg) | Generado |
| popular_8_bv_5_vq | 5 | 173 → 16 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_8_bv_5_vq.svg) | Generado |
| popular_9_bv_9_vq | 9 | 329 → 29 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_9_bv_9_vq.svg) | Generado |
| popular_10_bv_14_vq | 14 | 524 → 44 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_10_bv_14_vq.svg) | Generado |
| popular_11_dj_indep_4_mqt | 4 | 120 → 14 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_11_dj_indep_4_mqt.svg) | Generado |
| popular_12_dj_indep_5_mqt | 5 | 157 → 17 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_12_dj_indep_5_mqt.svg) | Generado |
| popular_13_dj_indep_7_mqt | 7 | 233 → 23 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_13_dj_indep_7_mqt.svg) | Generado |
| popular_14_dj_indep_14_mqt | 14 | 496 → 43 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_14_dj_indep_14_mqt.svg) | Generado |
| popular_15_grover-noancilla_3_mqt | 3 | 171 → 22 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_15_grover-noancilla_3_mqt.svg) | Generado |
| popular_16_grover-noancilla_4_mqt | 4 | 880 → 113 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_16_grover-noancilla_4_mqt.svg) | Generado |
| popular_17_grover-v-chain_3_mqt | 3 | 171 → 22 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_17_grover-v-chain_3_mqt.svg) | Generado |
| popular_18_grover-v-chain_4_mqt | 4 | 880 → 113 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_18_grover-v-chain_4_mqt.svg) | Generado |
| popular_19_grover_3_vq | 3 | 214 → 37 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_19_grover_3_vq.svg) | Generado |
| popular_20_grover_7_vq | 7 | 1022 → 152 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_20_grover_7_vq.svg) | Generado |
| popular_21_grover_9_vq | 9 | 1495 → 207 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_21_grover_9_vq.svg) | Generado |
| popular_22_kickback_7_vq | 7 | 188 → 28 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_22_kickback_7_vq.svg) | Generado |
| popular_23_pe_2_vq | 3 | 83 → 17 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_23_pe_2_vq.svg) | Generado |
| popular_24_pe_3_mqt | 3 | 75 → 14 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_24_pe_3_mqt.svg) | Generado |
| popular_25_pe_4_mqt | 4 | 125 → 20 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_25_pe_4_mqt.svg) | Generado |
| popular_26_pe_5_mqt | 5 | 257 → 42 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_26_pe_5_mqt.svg) | Generado |
| popular_27_pe_6_mqt | 6 | 345 → 56 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_27_pe_6_mqt.svg) | Generado |
| popular_28_pe_7_mqt | 7 | 458 → 72 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_28_pe_7_mqt.svg) | Generado |
| popular_29_qft_2_mqt | 2 | 39 → 10 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_29_qft_2_mqt.svg) | Generado |
| popular_30_qft_3_mqt | 3 | 84 → 18 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_30_qft_3_mqt.svg) | Generado |
| popular_31_qft_4_mqt | 4 | 154 → 32 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_31_qft_4_mqt.svg) | Generado |
| popular_32_qft_6_mqt | 6 | 345 → 58 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_32_qft_6_mqt.svg) | Generado |
| popular_33_qft_7_vq | 7 | 448 → 79 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_33_qft_7_vq.svg) | Generado |
| popular_34_shor_mod15_mqt | 5 | 288 → 11 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_34_shor_mod15_mqt.svg) | Generado |
| popular_35_shor_mod21_mqt | 8 | 245 → 28 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_35_shor_mod21_mqt.svg) | Generado |
| popular_36_simon_vq | 6 | 165 → 20 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_36_simon_vq.svg) | Generado |
| popular_37_tsp_indep_4_mqt | 4 | 394 → 40 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_37_tsp_indep_4_mqt.svg) | Generado |
| popular_38_tsp_indep_5_mqt | 5 | 798 → 109 | [Abrir SVG](../output/run_14_10_2026__20/ZXCalculus/graphs/popular_38_tsp_indep_5_mqt.svg) | Generado |


SVG generados: 53/53; errores: 0.
Archivos guardados en: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20\ZXCalculus\graphs


# Migración de circuitos a ecosistemas / lenguajes concretos


### Generación de circuitos cuánticos a partir del código OpenQASM

Dos pasos, en celdas separadas:

1. **Carga**: lee los `.txt` de `output/run_DD_MM_YYYY__HH/algorithms_qasm` y hace el `loads()` del OpenQASM por proveedor con `cargar_circuito_qasm()` (`utils/qutils.py`), dejando los circuitos en el diccionario `circuitos_cargados`.
2. **Generación**: a partir de esa carga, `provider_python_source()` crea un script `.py` por circuito y proveedor.

Los proveedores salen del `.env`: `QTRANS_PROVIDERS` (o la clave legada `QTRANS_TARGET_PROVIDERS`), con valores `qiskit`, `braket` y `pennylane` (alias: `ibm`, `aws`, `xanadu`).

Salida: `output/run_DD_MM_YYYY__HH/migrated_circuits/<subcarpeta>/`, con `ibm_qiskit/`, `aws_braket/` y `xanadu_pennylane/`.

In [11]:
from pathlib import Path
import os
import re

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from utils.qutils import PROVIDER_DIRECTORY, cargar_circuito_qasm, providers_from_env

if 'RUN_QASM_DIR' not in globals():
    raise RuntimeError('Faltan las rutas de la corrida: ejecutá primero la celda de configuración (PROJECT_ROOT / RUN_*).')

if 'strip_qasm_version' not in globals():
    def strip_qasm_version(stem):
        return re.sub(r'_v\d+\.\d+$', '', stem)


def _env_flag(name, default=True):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().casefold() in {'1', 'true', 'yes', 'on'}


load_dotenv()
QASM_VERSION = (os.getenv("QTRANS_QASM_VERSION") or globals().get('QASM_VERSION') or '3.0').strip()
if QASM_VERSION not in ('2.0', '3.0'):
    raise ValueError(f"QASM_VERSION no válido: {QASM_VERSION!r} (use '2.0' o '3.0')")
QASM_VERSION_SUFFIX = f'_v{QASM_VERSION}'

SHOTS = int(os.getenv('QTRANS_SHOTS') or 1000)
PRESERVE_CUSTOM_GATES = _env_flag('QTRANS_PRESERVE_CUSTOM_GATES', True)

# Proveedores del .env: QTRANS_PROVIDERS (preferido) o QTRANS_TARGET_PROVIDERS (legado).
PROVIDERS = providers_from_env()

qasm_dir = RUN_QASM_DIR
if not qasm_dir.is_dir():
    # Respaldo: si esta corrida aún no tiene OpenQASM, se leen los archivos del
    # run más reciente que sí los tenga (se avisa para no abortar por la hora).
    candidatos = sorted(
        (
            candidate / 'algorithms_qasm'
            for candidate in RUN_DIR.parent.glob('run_*')
            if (candidate / 'algorithms_qasm').is_dir()
        ),
        key=lambda candidate: candidate.parent.stat().st_mtime,
        reverse=True,
    )
    if candidatos:
        qasm_dir = candidatos[0]
        print(
            f'⚠️ No hay OpenQASM en {RUN_DIR.name}; '
            f'se leen los archivos de {qasm_dir.parent.name}.'
        )
    else:
        raise FileNotFoundError(
            f'No existe la carpeta de OpenQASM: {RUN_QASM_DIR} (ni en ningún run_*). '
            'Ejecutá antes la celda "Generación de circuitos OpenQASM".'
        )

qasm_files = sorted(
    (
        path for path in qasm_dir.iterdir()
        if path.is_file() and path.suffix.lower() == '.txt'
        and path.name.endswith(QASM_VERSION_SUFFIX + '.txt')
    ),
    key=lambda path: path.name,
)
if not qasm_files:
    raise FileNotFoundError(
        f'No se encontraron archivos OpenQASM {QASM_VERSION} '
        f'(*{QASM_VERSION_SUFFIX}.txt) en {qasm_dir}'
    )

qasm_sources = {
    strip_qasm_version(path.stem): path.read_text(encoding='utf-8')
    for path in qasm_files
}

print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')
print(f'VERSIÓN DE OpenQASM procesada: QASM {QASM_VERSION}')
print(f'PROVEEDORES: {", ".join(PROVIDERS)}')

# loads() del OpenQASM por proveedor (utils.qutils.cargar_circuito_qasm).
circuitos_cargados = {provider: {} for provider in PROVIDERS}
carga_filas = []
carga_errores = []

for provider in PROVIDERS:
    for name, qasm_source in qasm_sources.items():
        try:
            circuito = cargar_circuito_qasm(
                qasm_source,
                provider,
                qasm_version=QASM_VERSION,
                shots=SHOTS,
                preserve_custom_gates=PRESERVE_CUSTOM_GATES,
            )
        except Exception as exc:
            carga_errores.append((provider, name, exc))
            print(f'No se pudo cargar {provider}/{name} ({type(exc).__name__}): {exc!r}')
            continue
        circuitos_cargados[provider][name] = circuito
        carga_filas.append({
            'Proveedor': provider,
            'Circuito': name,
            'Tipo': type(circuito).__name__,
        })

df_carga = pd.DataFrame(carga_filas, columns=['Proveedor', 'Circuito', 'Tipo'])
display(df_carga)

print('\nℹ️ Circuitos cargados por proveedor:')
for provider in PROVIDERS:
    cargados = len(circuitos_cargados[provider])
    subcarpeta = PROVIDER_DIRECTORY.get(provider, provider)
    estado = 'OK' if cargados == len(qasm_sources) else 'INCOMPLETO'
    print(f'- {provider} (migrated_circuits/{subcarpeta}): '
          f'{cargados}/{len(qasm_sources)} circuitos [{estado}]')

if carga_errores:
    print(f'\n⚠️ {len(carga_errores)} circuitos no se pudieron cargar (detalle arriba).')
print(f'\nℹ️ {len(qasm_sources)} archivos OpenQASM leídos desde: {qasm_dir.resolve()}')


DIRECTORIO DE LA CORRIDA: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20
VERSIÓN DE OpenQASM procesada: QASM 3.0
PROVEEDORES: qiskit, braket, pennylane


c:\Users\PC\desarrollo\QTransLIFIA\.venv\Lib\site-packages\pennylane\devices\device_api.py:207: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(
c:\Users\PC\desarrollo\QTransLIFIA\.venv\Lib\site-packages\pennylane\devices\device_api.py:207: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(
c:\Users\PC\desarrollo\QTransLIFIA\.venv\Lib\site-packages\pennylane\devices\device_api.py:207: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


,Proveedor,Circuito,Tipo
0,qiskit,10__QAOA,QuantumCircuit
1,qiskit,11__Kickback,QuantumCircuit
2,qiskit,12__Full_Adder,QuantumCircuit
3,qiskit,13__Multicontroled_Gates,QuantumCircuit
4,qiskit,14__Shor_6q,QuantumCircuit
...,...,...,...
154,pennylane,popular_5__bv_2_vq,QNode
155,pennylane,popular_6__bv_3_vq,QNode
156,pennylane,popular_7__bv_4_vq,QNode
157,pennylane,popular_8__bv_5_vq,QNode



ℹ️ Circuitos cargados por proveedor:
- qiskit (migrated_circuits/ibm_qiskit): 53/53 circuitos [OK]
- braket (migrated_circuits/aws_braket): 53/53 circuitos [OK]
- pennylane (migrated_circuits/xanadu_pennylane): 53/53 circuitos [OK]

ℹ️ 53 archivos OpenQASM leídos desde: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20\algorithms_qasm


In [12]:
from pathlib import Path
import os
import re

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from utils.qutils import PROVIDER_DIRECTORY, provider_python_source

if 'RUN_CIRCUITS_DIR' not in globals():
    raise RuntimeError('Faltan las rutas de la corrida: ejecutá primero la celda de configuración (PROJECT_ROOT / RUN_*).')
if 'qasm_sources' not in globals() or 'circuitos_cargados' not in globals():
    raise RuntimeError('Falta la carga previa: ejecutá la celda anterior (cargar_circuito_qasm).')
if 'PROVIDERS' not in globals():
    raise RuntimeError('Falta la definición de PROVIDERS: ejecutá la celda anterior.')

if 'strip_qasm_version' not in globals():
    def strip_qasm_version(stem):
        return re.sub(r'_v\d+\.\d+$', '', stem)


def _env_flag(name, default=True):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().casefold() in {'1', 'true', 'yes', 'on'}


load_dotenv()
QASM_VERSION = (os.getenv("QTRANS_QASM_VERSION") or globals().get('QASM_VERSION') or '3.0').strip()
if QASM_VERSION not in ('2.0', '3.0'):
    raise ValueError(f"QASM_VERSION no válido: {QASM_VERSION!r} (use '2.0' o '3.0')")
QASM_VERSION_SUFFIX = f'_v{QASM_VERSION}'

SHOTS = int(os.getenv('QTRANS_SHOTS') or 1000)
PRESERVE_CUSTOM_GATES = _env_flag('QTRANS_PRESERVE_CUSTOM_GATES', True)

migrated_root = RUN_CIRCUITS_DIR
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

generacion_filas = []
generacion_errores = []

for provider in PROVIDERS:
    if provider not in circuitos_cargados:
        continue
    subcarpeta = PROVIDER_DIRECTORY.get(provider, provider)
    provider_dir = migrated_root / subcarpeta
    provider_dir.mkdir(parents=True, exist_ok=True)

    for name, qasm_source in qasm_sources.items():
        if name not in circuitos_cargados[provider]:
            # Solo se migra lo que la celda anterior cargó para ese proveedor.
            continue
        try:
            python_source = provider_python_source(
                qasm_source,
                provider,
                qasm_version=QASM_VERSION,
                shots=SHOTS,
                preserve_custom_gates=PRESERVE_CUSTOM_GATES,
            )
            output_path = provider_dir / f'{name}.py'
            compile(python_source, str(output_path), 'exec')
            output_path.write_text(python_source, encoding='utf-8')
        except Exception as exc:
            generacion_errores.append((provider, name, exc))
            print(f'Error generando {provider}/{name} ({type(exc).__name__}): {exc!r}')
            continue

        generacion_filas.append({
            'Proveedor': provider,
            'Script': f'{subcarpeta}/{output_path.name}',
            'Líneas': len(python_source.splitlines()),
        })

df_generacion = pd.DataFrame(generacion_filas, columns=['Proveedor', 'Script', 'Líneas'])
display(df_generacion)

print('\nℹ️ Scripts migrados por proveedor:')
for provider in PROVIDERS:
    subcarpeta = PROVIDER_DIRECTORY.get(provider, provider)
    generados = sum(1 for fila in generacion_filas if fila['Proveedor'] == provider)
    print(f'- {provider} -> migrated_circuits/{subcarpeta}: {generados} scripts')

if generacion_errores:
    print(f'\n⚠️ {len(generacion_errores)} scripts con error (detalle arriba).')
print(f'\nℹ️ Carpeta de circuitos migrados: {migrated_root.resolve()}')


DIRECTORIO DE LA CORRIDA: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20


,Proveedor,Script,Líneas
0,qiskit,ibm_qiskit/10__QAOA.py,15
1,qiskit,ibm_qiskit/11__Kickback.py,19
2,qiskit,ibm_qiskit/12__Full_Adder.py,23
3,qiskit,ibm_qiskit/13__Multicontroled_Gates.py,28
4,qiskit,ibm_qiskit/14__Shor_6q.py,42
...,...,...,...
154,pennylane,xanadu_pennylane/popular_5__bv_2_vq.py,15
155,pennylane,xanadu_pennylane/popular_6__bv_3_vq.py,25
156,pennylane,xanadu_pennylane/popular_7__bv_4_vq.py,30
157,pennylane,xanadu_pennylane/popular_8__bv_5_vq.py,35



ℹ️ Scripts migrados por proveedor:
- qiskit -> migrated_circuits/ibm_qiskit: 53 scripts
- braket -> migrated_circuits/aws_braket: 53 scripts
- pennylane -> migrated_circuits/xanadu_pennylane: 53 scripts

ℹ️ Carpeta de circuitos migrados: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20\migrated_circuits


### Generación de circuitos utilizando IBM Composer

Recorre los algoritmos migrados en `migrated_circuits/ibm_qiskit` y, a partir de su código OpenQASM, genera en `migrated_circuits/ibm_qiskit/imgs` una imagen `.png` por circuito con el estilo de IBM Quantum Composer (render local con Qiskit, `circuit.draw(output='mpl', style='iqp')`).

In [13]:
from pathlib import Path
import os
import re
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, qasm3
from dotenv import load_dotenv

if 'strip_qasm_version' not in globals():
    def strip_qasm_version(stem):
        return re.sub(r'_v\d+\.\d+$', '', stem)

# Reutiliza la configuración de la corrida; si la celda se ejecuta aislada, la reconstruye
if 'RUN_CIRCUITS_DIR' not in globals() or 'RUN_QASM_DIR' not in globals():
    from datetime import datetime

    def _find_output_root():
        _root = globals().get('PROJECT_ROOT') or Path.cwd()
        for _candidate in (_root / 'output', Path.cwd() / 'output', Path.cwd().parent / 'output'):
            if _candidate.is_dir():
                return _candidate
        return _root / 'output'

    _output_root = _find_output_root()
    _existing_runs = sorted(
        (
            d for d in _output_root.glob('run_*')
            if (d / 'migrated_circuits' / 'ibm_qiskit').is_dir()
            or (d / 'migrated_circuits' / 'qiskit').is_dir()  # corridas anteriores
        ),
        key=lambda d: d.stat().st_mtime,
        reverse=True,
    )
    RUN_DIR = _existing_runs[0] if _existing_runs else _output_root / f"run_{datetime.now().strftime('%d_%m_%Y__%H')}"
    RUN_QASM_DIR = RUN_DIR / 'algorithms_qasm'
    RUN_CIRCUITS_DIR = RUN_DIR / 'migrated_circuits'

load_dotenv()
QASM_VERSION = (os.getenv("QTRANS_QASM_VERSION") or globals().get('QASM_VERSION') or '3.0').strip()
if QASM_VERSION not in ('2.0', '3.0'):
    raise ValueError(f"QASM_VERSION no válido: {QASM_VERSION!r} (use '2.0' o '3.0')")
QASM_VERSION_SUFFIX = f'_v{QASM_VERSION}'

qasm_dir = RUN_QASM_DIR
# Los scripts de Qiskit viven en migrated_circuits/ibm_qiskit (nombre legado: 'qiskit').
qiskit_subdir = 'ibm_qiskit' if (RUN_CIRCUITS_DIR / 'ibm_qiskit').is_dir() else 'qiskit'
qiskit_dir = RUN_CIRCUITS_DIR / qiskit_subdir
images_dir = qiskit_dir / 'imgs'
images_dir.mkdir(parents=True, exist_ok=True)
print(f'DIRECTORIO DE LA CORRIDA: {RUN_DIR.resolve()}')

if not qiskit_dir.is_dir():
    raise FileNotFoundError(
        f'No existe la carpeta de circuitos migrados a Qiskit: {qiskit_dir}. '
        'Ejecutá antes la celda de generación y verificá que QTRANS_PROVIDERS incluya qiskit.'
    )


def natural_sort_key(value):
    parts = re.split(r'[_\s.-]+', str(value))
    return [(0, int(part)) if part.isdigit() else (1, part.lower()) for part in parts if part]


def load_qasm_circuit(qasm_text):
    text = (qasm_text or '').strip()
    upper = text.upper()
    if upper.startswith('OPENQASM 2.0'):
        return QuantumCircuit.from_qasm_str(text)
    if upper.startswith('OPENQASM 3.0'):
        return qasm3.loads(text)
    try:
        return qasm3.loads(text)
    except Exception:
        return QuantumCircuit.from_qasm_str(text)


migrated_scripts = sorted(qiskit_dir.glob('*.py'), key=lambda path: natural_sort_key(path.stem))
if not migrated_scripts:
    raise FileNotFoundError(f'No hay scripts migrados (.py) en {qiskit_dir}')

rendered = 0
failed = 0
for script_path in migrated_scripts:
    name = strip_qasm_version(script_path.stem)
    qasm_path = qasm_dir / f'{name}{QASM_VERSION_SUFFIX}.txt'
    if not qasm_path.is_file():
        print(f'Sin OpenQASM asociado para {script_path.name}: {qasm_path.name}')
        failed += 1
        continue

    png_path = images_dir / f'{name}.png'
    figure = None
    try:
        circuit = load_qasm_circuit(qasm_path.read_text(encoding='utf-8'))
        figure = circuit.draw(output='mpl', filename=str(png_path), style='iqp')
        print(f'Generado: {png_path.name}')
        rendered += 1
    except Exception as exc:
        print(f'No se pudo dibujar {script_path.name} ({type(exc).__name__}): {exc!r}')
        failed += 1
    finally:
        if figure is not None:
            plt.close(figure)

print(f'\nℹ️ Resumen: {rendered}/{len(migrated_scripts)} imágenes generadas; {failed} archivos con error.')
print(f'Imágenes (estilo IBM Quantum Composer) guardadas en: {images_dir.resolve()}')


DIRECTORIO DE LA CORRIDA: C:\Users\PC\desarrollo\QTransLIFIA\output\run_14_10_2026__20
Generado: 1__Shor.png
Generado: 2__Bernstein-Vazirani.png
Generado: 3__Grover.png
Generado: 4__Deutsch-Jozsa.png
Generado: 5__Simon.png
Generado: 6__TSP.png
Generado: 7__Teleportation.png
Generado: 8__Phase_Estimation.png
Generado: 9__QFT.png
Generado: 10__QAOA.png
Generado: 11__Kickback.png
Generado: 12__Full_Adder.png
Generado: 13__Multicontroled_Gates.png
Generado: 14__Shor_6q.png
Generado: 15__Popurri_6q.png
Generado: popular_1__adder_n4_vq.png
Generado: popular_2__adder_n10_vq.png
Generado: popular_3__adder_n13_vq.png
Generado: popular_4__adder_n16_vq.png
Generado: popular_5__bv_2_vq.png
Generado: popular_6__bv_3_vq.png
Generado: popular_7__bv_4_vq.png
Generado: popular_8__bv_5_vq.png
Generado: popular_9__bv_9_vq.png
Generado: popular_10__bv_14_vq.png
Generado: popular_11__dj_indep_4_mqt.png
Generado: popular_12__dj_indep_5_mqt.png
Generado: popular_13__dj_indep_7_mqt.png
Generado: popular_14__d

### Obtención de url para Quirk a partir de un json register

Utilidad de prueba: a partir de un registro de algoritmo definido como variable string (json), genera la URL de Quirk codificada y funcional para "https://algassert.com/quirk", validando antes que el circuito se decodifica correctamente (round-trip).


In [14]:
# ----------------------------------------------------------------------------
# Utilidad de prueba: recibe directamente la URL cruda de Quirk (aunque venga
# mal formateada, con "\" y "\"" de más) y la limpia/formatea para que el
# circuito se vea correctamente en https://algassert.com/quirk.
# ----------------------------------------------------------------------------

import re

from IPython.display import HTML, display

from utils.qutils import quirk_circuit_info
from utils.url_utils import encode_quirk_url, parse_quirk_url

# URL cruda del circuito (mal formateada a propósito: backslashes y comillas)
url_quirk_cruda = (
    '"https://algassert.com/quirk#circuit={\"cols\":[[1,\"•\",\"•\",\"X\",1,1,1,1,1,1,1,1,1],[1,\"•\",\"X\",1,1,1,1,1,1,1,1,1,1],[1,1,1,1,\"•\",\"•\",\"X\",1,1,1,1,1,1],[1,1,1,1,\"•\",\"X\",1,1,1,1,1,1,1],[1,1,1,1,1,1,1,\"•\",\"•\",\"X\",1,1,1],[1,1,1,1,1,1,1,\"•\",\"X\",1,1,1,1],[1,1,1,1,1,1,1,1,1,1,\"•\",\"•\",\"X\"],[1,1,1,1,1,1,1,1,1,1,\"•\",\"X\",1],[\"•\",1,\"•\",\"X\",1,1,1,1,1,1,1,1,1],[1,1,1,\"•\",1,\"•\",\"X\",1,1,1,1,1,1],[1,1,1,1,1,1,\"•\",1,\"•\",\"X\",1,1,1],[1,1,1,1,1,1,1,1,1,\"•\",1,\"•\",\"X\"],[\"•\",1,\"X\",1,1,1,1,1,1,1,1,1,1],[1,1,1,\"•\",1,\"X\",1,1,1,1,1,1,1],[1,1,1,1,1,1,\"•\",1,\"X\",1,1,1,1],[1,1,1,1,1,1,1,1,1,\"•\",1,\"X\",1],[\"Measure\",1,1,1,1,1,1,1,1,1,1,1,1],[1,\"Measure\",1,1,1,1,1,1,1,1,1,1,1],[1,1,\"Measure\",1,1,1,1,1,1,1,1,1,1],[1,1,1,\"Measure\",1,1,1,1,1,1,1,1,1],[1,1,1,1,\"Measure\",1,1,1,1,1,1,1,1],[1,1,1,1,1,\"Measure\",1,1,1,1,1,1,1],[1,1,1,1,1,1,\"Measure\",1,1,1,1,1,1],[1,1,1,1,1,1,1,\"Measure\",1,1,1,1,1],[1,1,1,1,1,1,1,1,\"Measure\",1,1,1,1],[1,1,1,1,1,1,1,1,1,\"Measure\",1,1,1],[1,1,1,1,1,1,1,1,1,1,\"Measure\",1,1],[1,1,1,1,1,1,1,1,1,1,1,\"Measure\",1],[1,1,1,1,1,1,1,1,1,1,1,1,\"Measure\"]]}"'
)


def limpiar_url_quirk(url):
    """Normaliza una URL cruda: quita comillas externas y backslashes sobrantes."""
    if not isinstance(url, str):
        raise ValueError("La URL debe ser una cadena.")
    s = url.strip()
    # Quitar comillas que envuelven toda la cadena
    if len(s) >= 2 and s[0] == s[-1] and s[0] in ('"', "'"):
        s = s[1:-1]
    # Desescapar \"  \'  y eliminar cualquier backslash sobrante
    s = re.sub(r'\\(["\'])', r'\1', s)
    s = s.replace('\\', '')
    s = s.strip().strip('"').strip("'").strip()
    # Si viene con prefijo de registro ("url": ...), cortar desde https://
    idx = s.find('https://')
    if idx > 0:
        s = s[idx:]
    return s


def formatear_url_quirk(url):
    """Acepta la URL cruda (aunque traiga \\ y \" extra) y la percent-encodea."""
    limpia = limpiar_url_quirk(url)
    if "circuit=" not in limpia:
        raise ValueError("La URL debe contener 'circuit=' con el JSON del circuito.")
    return encode_quirk_url(limpia)


def validar_url_quirk(final_url):
    """Valida round-trip: decodifica la URL y reporta info del circuito."""
    try:
        circuito = parse_quirk_url(final_url)
    except Exception:
        print("Estado      : ERROR (no se pudo parsear el circuito)")
        return False
    if isinstance(circuito, dict) and "cols" in circuito:
        info = quirk_circuit_info(final_url)
        print(f"Estado      : OK ({info['n_qubits']} qubits, {info['n_cols']} columnas)")
        return True
    print("Estado      : ERROR (no se pudo parsear el circuito)")
    return False


def mostrar_url_quirk(url):
    """Limpia/formatea la URL recibida y muestra el enlace 'Abrir en Quirk'."""
    final_url = formatear_url_quirk(url)
    validar_url_quirk(final_url)
    display(HTML('<a href="' + final_url + '" target="_blank">Abrir en Quirk</a>'))
    return final_url


# --- Prueba: recibe directamente la URL cruda del circuito -------------------
mostrar_url_quirk(url_quirk_cruda)


Estado      : OK (13 qubits, 29 columnas)


'https://algassert.com/quirk#circuit=%7B%22cols%22%3A%5B%5B1%2C%22%E2%80%A2%22%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22%E2%80%A2%22%2C%22X%22%5D%2C%5B1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%5D%2C%5B%22%E2%80%A2%22%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C%22%E2%80%A2%22%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2C1%2C1%2C1%2C1%2C1%5D%2C%5B1%2C1%2C1%2C1%2C1%2C1%2C%22%E2%80%A2%22%2C1%2C%22%E2%80%A2%22%2C%22X%22%2C1%2